# Distinguish runtime memory and packed storage

The simulator stores floating tensors. Ideal packed bytes and operation counts are theoretical quantities; measured runtime belongs to the actual host backend.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / "edgeformer").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "edgeformer").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
torch.set_num_threads(1)
torch.manual_seed(42)
from edgeformer.models import TinyViT

model = TinyViT(
    image_size=16,
    patch_size=4,
    num_classes=4,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
)

In [ ]:
from edgeformer.profiling import estimate_vit_macs, storage_report, benchmark_latency
from edgeformer.quantization import quantize_model_weights

model.eval()
sample = torch.randn(1, 3, 16, 16)
rows = []
for bits in [32, 8, 4]:
    variant = quantize_model_weights(model, bits).eval()
    report = storage_report(variant, bits)
    timing = benchmark_latency(variant, sample, warmup=2, runs=10, return_stats=True)
    rows.append(
        {
            "bits": bits,
            "runtime_bytes": report["runtime_parameter_bytes"],
            "ideal_packed_bytes": report["ideal_packed_total_bytes"],
            "median_ms": timing["median_ms"],
        }
    )
print("Per-image MACs:", estimate_vit_macs(model)["total"])
rows

In [ ]:
import matplotlib.pyplot as plt

labels = [f"W{r['bits']}" for r in rows]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar(labels, [r["runtime_bytes"] / 1024 for r in rows], label="runtime tensors")
axes[0].bar(labels, [r["ideal_packed_bytes"] / 1024 for r in rows], alpha=0.7, label="ideal packed")
axes[0].set_ylabel("KiB")
axes[0].legend()
axes[1].bar(labels, [r["median_ms"] for r in rows])
axes[1].set_ylabel("Floating forward latency (ms/batch)")
fig.tight_layout()

To inspect a matched trained-model sweep, regenerate `examples/results/` with `python scripts/reproduce_examples.py`. Its accuracy/storage Pareto set does not rank noisy floating host timings as hardware results.